# 11 Execution Readiness Checklist

This notebook checks whether the repo structure is aligned for later execution without launching the models now.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [ ]:
paths_to_check = [
    PACKAGE_ROOT / "run_naive_benchmark.py",
    PACKAGE_ROOT / "run_lear_benchmark.py",
    PACKAGE_ROOT / "run_xgboost_benchmark.py",
    PACKAGE_ROOT / "run_prophet_benchmark.py",
    PACKAGE_ROOT / "run_model_comparison.py",
    PACKAGE_ROOT / "docs" / "feature_sets.md",
    PACKAGE_ROOT / "docs" / "tuning_policy.md",
    PACKAGE_ROOT / "hourly_da" / "core" / "methodology.py",
    PACKAGE_ROOT / "hourly_da" / "core" / "tuning.py",
    PACKAGE_ROOT / "hourly_da" / "models" / "registry.py",
    PACKAGE_ROOT / "hourly_da" / "models" / "prophet_model.py",
]

check_rows = []
for path in paths_to_check:
    check_rows.append({"path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()})

display(pd.DataFrame(check_rows))


In [ ]:
rows = []
for run_label in ['case_week_selection', 'naive_benchmark', 'lear_benchmark', 'xgboost_benchmark', 'prophet_benchmark', 'model_comparison']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


If the files above exist, the methodology tables are centralized, and the run labels are recognized, the repo is structurally ready for the later benchmark phase.
